In [ ]:
import sys
import math
from random import randint
import pandas as pd
from PySide6 import QtGui, QtWidgets, QtCore
from PySide6.QtCore import QThread, Signal, QMutex, QMutexLocker
from PySide6.QtWidgets import QApplication, QMainWindow, QDialog, QLabel, QVBoxLayout, QWidget, QGraphicsPathItem, QMessageBox, QErrorMessage
from PySide6.QtCore import QTimer 
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore  
import numpy as np
import serial
import serial.tools.list_ports
import time
from live_plotting_ui import Ui_MainWindow

Serial Reader Thread

In [ ]:
class SerialReaderThread(QThread):
    data_received = Signal(float, float) #Signal for right and left forces
    error_message_signal = Signal(str, str)  # Signal for error message


    def __init__(self):
        super().__init__()

        self.baudrate = 115200
        self.running = False
        self.serial = None # initialize as none
        self.com_port = None

        # global file_path
        # file_path = None

        # Calibration factors for each channel
        self.calibration_factors = {
            '1A': 0.05053931524,  
            '1B': 0.05109740826,
            '1C': 0.05108326319,
            '1D': 0.05021199964,
            '2A': 0.04986771817,
            '2B': 0.05140285445,
            '2C': 0.05292458815,
            '2D': 0.05131591832}
        # Initialize data
        self.raw_ADC_vals = {}
        self.channel_force_vals = {}
        self.tared_channel_values = {}
        self.channel_tares = {'1A':0, '1B':0, '1C':0, '1D':0,
        '2A':0, '2B':0, '2C':0, '2D':0}
        self.force_left = 0
        self.force_right = 0
        self.force_left_list = []
        self.force_right_list = []
        self.no_data_start_time = None  # Track the start time when no data is received
        self.error_threshold = 1  # Time threshold in seconds

        self.last_emit_time = 0  # Time in seconds

        # self.saving_data = False
        # self.last_write = None
        # self.new_tares = False

    def autodetect_com_port(self):
        """Auto-detect the Raspberry Pi Pico's COM port."""
        ports = serial.tools.list_ports.comports()
        for port in ports:
            if "USB" in port.description:
                self.com_port = port.device
                print(f"Port found: {self.com_port}")
                return True
        print("No Raspberry Pi Pico found.")
        return False

    def run(self):
        """Thread execution starts here."""
        if not self.autodetect_com_port():
            return

        try:
            self.serial = serial.Serial(self.com_port, self.baudrate, timeout=1)
            print("serial port detected")
            self.running = True

            while self.running:
                try:
                    # Check if we have any data to read, and make sure the connection is valid
                    if self.serial.in_waiting:
                        # print("line in waiting")
                        line = self.serial.readline().decode('utf-8').strip()
                        if line:
                            channel, value = line.split(':')
                            channel = channel.strip()
                            value = int(value.strip())
                            self.update_channel_value(channel, value)

                            self.no_data_start_time = None
                    else:
                        # If there's no data but the serial connection is active, check if it's open
                        if not self.serial.is_open:
                            raise serial.SerialException("Connection lost during read.")
                        else:
                            if self.no_data_start_time is None:
                                self.no_data_start_time = time.time()
                            if time.time() - self.no_data_start_time >= self.error_threshold:
                                self.error_message_signal.emit("Serial Error", "Not receiving data from pico. Unplug and replug the USB code, and restart the GUI program.")
                                break
                except serial.SerialException as e:
                    # Handle serial connection loss or any other serial error
                    print(f"Serial Error: {e}")
                    self.error_message_signal.emit("Serial Error", f"Connection lost: {e}")
                    self.running = False
                    break
                except Exception as e:
                    # Catch any other exceptions (e.g., parsing errors)
                    print(f"Unexpected error: {e}")
                    self.error_message_signal.emit("Unexpected Error", str(e))
                    self.running = False
                    break
        finally:
            # Ensure the serial connection is properly closed
            if self.serial and self.serial.is_open:
                self.serial.close()
                print("Serial connection closed safely.")

    def update_channel_value(self, channel, value):
        """Update values and compute force sums."""
        if channel in self.calibration_factors:
            self.raw_ADC_vals[channel] = round(value,6)
            calibrated_value = value * self.calibration_factors[channel]
            self.channel_force_vals[channel] = round(calibrated_value,6)
            self.tared_channel_values[channel] = round(calibrated_value - self.channel_tares[channel],6)
            self.compute_sums()

    def compute_sums(self):
        """Compute the sum of values for left and right force sensors."""
        self.force_right = round(sum(value for channel, value in self.tared_channel_values.items() if channel[0] == '1'), 4)
        self.force_left = round(sum(value for channel, value in self.tared_channel_values.items() if channel[0] == '2'), 4)
         # if self.saving_data:
        #     if (self.last_write == None) or (time.time() - self.last_write >= 0.001):
        #         self.write_csv_row()
        
        current_time = time.time()
        if current_time - self.last_emit_time >= 0.01:  # Emit only every 10 ms
            self.data_received.emit(self.force_right, self.force_left)
            self.last_emit_time = current_time
            # print statement can be used to monitor connection (optional)
            # print(self.force_left)

    def filter_recent_data(self, data_list, current_time, time_window):
        """Filter list to keep only the last 10 seconds of data."""
        return [value for timestamp, value in data_list if current_time - timestamp <= time_window]
    
    def tare(self):
        self.channel_tares = self.channel_force_vals.copy()  # Ensure independent copy
        self.channel_tares = {k: round(v, 6) for k, v in self.channel_tares.items()}  # Reduce floating-point drift
        print("tare worked")
        # self.new_tares = True

    # def start_data_collection(self):
    #     self.open_csv()
    #     self.saving_data = True

    # def stop_data_collection(self):
    #     self.saving_data = False
    
    # def open_csv(self):
    #     # checks if there's a file path, opens csv and writes header
    #     global file_path
    #     if file_path == None:
    #         self.show_error_message("Error", "No file path selected.")

    #         print("no file path selected")
    #     else:
    #         print(file_path)
    #         """
    #         Write a single row of data to a CSV file. If the file doesn't exist, it creates one.
            
    #         :param file_name: Full path of the CSV file.
    #         :param data_row: List of values to write as a row.
    #         """
    #         file_exists = os.path.isfile(file_path)  # Check if the file exists

    #         with open(file_path, mode='a', newline='') as file:  # Open in append mode
    #             writer = csv.writer(file)

    #             # Write the header only if the file doesn't exist
    #             if not file_exists:
    #                 writer.writerow(["Timestamp", "Force Left (lbs)", "Force Right (lbs)", 
    #                                  "Force RA (lbs)", "Force RB (lbs)", "Force RC (lbs)", "Force RD (lbs)", 
    #                                  "Force LA (lbs)", "Force LB (lbs)", "Force LC (lbs)", "Force LD (lbs)",
    #                                  "Raw RA (ADC)", "Raw RB (ADC)", "Raw RC (ADC)", "Raw RD (ADC)",
    #                                  "Raw LA (ADC)", "Raw LB (ADC)", "Raw LC (ADC)", "Raw LD (ADC)",
    #                                  "Tare RA (ADC)", "Tare RB (ADC)", "Tare RC (ADC)", "Tare RD (ADC)",
    #                                  "Tare LA (ADC)", "Tare LB (ADC)", "Tare LC (ADC)", "Tare LD (ADC)"])

    # def write_csv_row(self):

    #     with open(file_path, mode='a', newline='') as file:  # Open in append mode
    #         writer = csv.writer(file)
    #         row = [str(datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]), self.force_left, self.force_right]
    #         row.extend(self.tared_channel_values.values())
    #         row.extend(self.raw_ADC_vals.values())
    #         if self.new_tares:
    #             row.extend(self.channel_tares.values())
    #             self.new_tares = False
    #         writer.writerow(row)# Write the data 
    #         self.last_write = time.time()
    #         print(self.channel_tares.values())

    def show_error_message(self, title, message):
        """
        Display an error message using QMessageBox.
        
        :param title: The title of the error message box.
        :param message: The message to be displayed.
        """
        msg_box = QMessageBox()
        msg_box.setIcon(QMessageBox.Critical)
        msg_box.setWindowTitle(title)
        msg_box.setText(message)
        msg_box.exec_()

    def stop(self):
        """Stop the thread safely."""
        self.running = False
        if self.serial and self.serial.is_open:
            self.serial.close()


Absolute Bar Graph

In [ ]:
class absbar_MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        # set up central widget
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.layout = QVBoxLayout(self.central_widget)

        # create plot widget
        self.plot_widget = pg.PlotWidget()
        self.layout.addWidget(self.plot_widget)
        self.plot_widget.setBackground("w")
        self.plot_widget.setTitle("<span style='color: black; font-size: 20pt; font-weight: bold;'>Force by Leg</span>")
        forcestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        self.plot_widget.setLabel("left", "Force (lbs)", **forcestyles)
        self.plot_widget.setYRange(-100, 300)
        self.plot_widget.getAxis("bottom").setTicks([[(2, "Left"), (3, "Right")]])

        # Initialize force data
        self.force_left = 0
        self.force_right = 0

        # Initialize bar graph
        self.bar_graph = pg.BarGraphItem(x=[2, 3], height=[self.force_left, self.force_right], width = 0.5)
        self.plot_widget.addItem(self.bar_graph)

        #Set up plot timer
        self.plot_timer = QTimer(self)
        self.plot_update_freq = 24
        self.plot_timer.setInterval(1000 / self.plot_update_freq) 
        self.plot_timer.timeout.connect(self.update_plot)
        
    def start_plotting(self):
        self.plot_timer.start()

    def receive_data(self, force_right, force_left):
        self.force_right = force_right
        self.force_left = force_left
        # print(self.force_left)
        
    def update_plot(self):
        threshold = 20
        if self.force_left is not None and self.force_right is not None:
            # Update bar graph height with the new force data
            self.bar_graph.setOpts(height=[self.force_left, self.force_right])
            if self.force_left - self.force_right > threshold: 
                brushes = ['r', 'black']
                pens = brushes
            elif self.force_right - self.force_left > threshold: 
                brushes = ['black', 'r']
                pens = brushes
            else:
                brushes = ['g', 'g']
                pens = brushes
            self.bar_graph.setOpts(brushes=brushes, pens=pens)


Diff Line

In [ ]:
class diffline_MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        # Set up graph
        self.plot_graph = pg.PlotWidget()
        self.setCentralWidget(self.plot_graph)
        self.plot_graph.setBackground("w")
        self.plot_graph.setTitle("<span style='color: black; font-size: 20pt; font-weight: bold;'>Force Difference vs Time</span>")

         # Show grid
        self.plot_graph.showGrid(x=True, y=True)

        
        # Identify styles for axes
        timestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        forcestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        # self.plot_graph.setLabel("left", "Time (s)", **timestyles)
        self.plot_graph.setLabel("bottom", "Force (lbs)", **forcestyles)
        cm = pg.ColorMap([0.0, 1.0], ['b', 'orange'])
        pen = cm.getPen(span=(-0.05, 0.05), orientation='horizontal', width=4)

         # Initialize graph line
        self.line = self.plot_graph.plot([], [], pen=pen)

        # Set axes range
        self.plot_graph.setYRange(0, 10)
        self.plot_graph.setXRange(-100, 100)  # Set based on your force diff data
        self.right_label = pg.TextItem(text="Right", angle=0, anchor=(0.5, 0.5), )
        self.right_label.setColor('#FFA500')
        self.left_label = pg.TextItem(text="Left", angle=0, anchor=(0.5, 0.5))
        self.left_label.setColor('b')
        self.display_window = 10  # Duration in seconds to show on the graph

        # Add a timer to update the plot
        self.plot_timer = QtCore.QTimer()
        self.sample_rate = 24
        self.plot_timer.setInterval(1000/self.sample_rate)
        self.plot_timer.timeout.connect(self.update_plot)

        # Initialize data lists
        self.force_right_list = []
        self.force_left_list = []

    # Receive data from SerialReaderThread
    def receive_data(self, force_right, force_left):
        if force_right and force_left:            
            # Append new data with timestamps
            current_time = time.time()
            self.force_right_list.append((current_time, force_right))
            self.force_left_list.append((current_time, force_left))

            # Keep only the last 10 seconds of data
            self.force_right_list = [(t, val) for t, val in self.force_right_list if current_time - t <= self.display_window]
            self.force_left_list = [(t, val) for t, val in self.force_left_list if current_time - t <= self.display_window]

    def start_plotting(self):
        self.force_right_list = []
        self.force_left_list = []
        self.plot_timer.start()

    def update_plot(self):
        if len(self.force_left_list) > 0 and len(self.force_right_list) > 0:
        # Extract timestamps and values for plotting
            times_right = [t for t, _ in self.force_right_list]
            values_right = [val for _, val in self.force_right_list]
            times_left = [t for t, _ in self.force_left_list]
            values_left = [val for _, val in self.force_left_list]

            # Ensure times are sorted
            times_right, values_right = zip(*sorted(zip(times_right, values_right)))
            times_left, values_left = zip(*sorted(zip(times_left, values_left)))

            self.asym = [r - l for r,l in zip(values_right,values_left)]

            # Set y-axis range to the last 'display_window' seconds
            start_time = max(times_right[0], times_left[0])
            end_time = times_right[-1] if times_right[-1] > times_left[-1] else times_left[-1]
            if end_time - start_time < 10:
                # keeps y-axis consistent even during first 10 seconds
                self.plot_graph.setYRange(start_time, start_time + 10) 
            else:
                self.plot_graph.setYRange(start_time, end_time)

            self.line.setData(self.asym, times_right)
            self.plot_graph.getAxis('left').setTicks([])

            # self.right_label.setPos(75, start_time + 1)
            # self.plot_graph.addItem(self.right_label)
            # self.left_label.setPos(-75, start_time + 1)
            # self.plot_graph.addItem(self.left_label)
            
            # if (values_right[0], values_left[0]) != 0:
            #     percent_asym = abs(self.asym[0]) / max(values_right[0], values_left[0])
            #     if (values_right[0] > 30) and (values_left[0] > 30) and (percent_asym > 0.1):
            #         self.plot_graph.setBackground('r')
            #     else: 
            #         self.plot_graph.setBackground('w')
            # else:
            #     self.plot_graph.setBackground('w')


Diff Bar

In [ ]:
class diffbar_MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        # set up central widget
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.layout = QVBoxLayout(self.central_widget)

        # create plot widget
        self.plot_widget = pg.PlotWidget()
        self.layout.addWidget(self.plot_widget)
        self.plot_widget.setBackground("w")
        self.plot_widget.setTitle("<span style='color: black; font-size: 20pt; font-weight: bold;'>Force Difference</span>")
        timestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        forcestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        self.plot_widget.setLabel("bottom", "Force (lbs)", **forcestyles)
        self.plot_widget.setRange(yRange=[-0.25, 1.5], xRange=[-100, 100])

        # set labels
        self.right_label = pg.TextItem(text="Right", angle=0, anchor=(0.5, 0.5))
        self.right_label.setPos(75, 1.25)  # Position at y=0.5 (adjust as necessary)
        self.right_label.setColor('#FFA500')
        self.plot_widget.addItem(self.right_label)
        self.left_label = pg.TextItem(text="Left", angle=0, anchor=(0.5, 0.5))
        self.left_label.setPos(-75, 1.25)  # Position at y=0.5 (adjust as necessary)
        self.left_label.setColor('b')
        self.plot_widget.addItem(self.left_label)

        # Hide the y-axis scale and labels
        self.plot_widget.getAxis('left').setTicks([])  # Remove tick marks
        self.plot_widget.getAxis('left').setStyle(showValues=False)  # Hide values

        #initialize bar graph
        self.bar_graph = pg.BarGraphItem(x0=0, width=0, height = 1, brushes=['b'])
        self.plot_widget.addItem(self.bar_graph)

        # Add a timer to simulate new temperature measurements
        self.plot_timer = QtCore.QTimer()
        self.sample_rate = 24
        self.plot_timer.setInterval(1000/self.sample_rate)
        self.plot_timer.timeout.connect(self.update_plot)

        #Initialize force data
        self.force_left = 0
        self.force_right = 0 

    def receive_data(self, force_right, force_left):
        self.force_right = force_right
        self.force_left = force_left

    def start_plotting(self):
        self.plot_timer.start()

    def update_plot(self):
        if self.force_left and self.force_right:
            self.asym = self.force_right - self.force_left
            if max(self.force_left,self.force_right) != 0:
                percent_asym = self.asym / max(self.force_left,self.force_right)
                
                # if (self.force_left > 30) & (self.force_right > 30) & (percent_asym > 0.1): 
                #     self.plot_widget.setBackground('r')
                # else: 
                #     self.plot_widget.setBackground('w')

                if self.asym > 0:
                    brush = ['#FFA500']
                else:
                    brush = ['b']
            else:
                    self.plot_widget.setBackground('w')
            self.bar_graph.setOpts(width=self.asym, brushes=brush)


Absolute Line Graph

In [ ]:
class AbsLineMainWindow(QMainWindow):
    def __init__(self):
        super().__init__()

        # Initialize plot widget
        self.plot_graph = pg.PlotWidget()
        self.setCentralWidget(self.plot_graph)
        self.plot_graph.setBackground("w")
        self.plot_graph.setTitle(
            "<span style='color: black; font-size: 20pt; font-weight: bold;'>Force vs Time</span>"
        )
        self.plot_graph.showGrid(x=True, y=True)
        timestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        forcestyles = {"color": "black", "font-size": "20px", "font-weight": "bold"}
        self.plot_graph.setLabel("left", "Force (lbs)", **forcestyles)
        self.plot_graph.getAxis('bottom').setTicks([])

        # self.plot_graph.setLabel("bottom", "Time", units=None, **timestyles)

        # Set axes range
        self.plot_graph.setXRange(0, 10)
        self.plot_graph.setYRange(-100, 225)
        self.display_window = 10  # Duration in seconds to show on the graph

        # Define pens for graph lines
        rightpen = pg.mkPen(color=(255, 165, 0), width=4)
        leftpen = pg.mkPen(color=(0, 0, 255), width=4)

        # Initialize graph lines
        self.right_line = self.plot_graph.plot([], [], name="Right", pen=rightpen)
        self.left_line = self.plot_graph.plot([], [], name="Left", pen=leftpen)

        # Add legend
        legend = pg.LegendItem(offset=(50, -50))
        legend.setParentItem(self.plot_graph.getViewBox())
        legend.addItem(self.right_line, "Right")
        legend.addItem(self.left_line, "Left")

        # Timer setup to update plot
        self.plot_timer = QTimer(self)
        self.plot_update_freq = 24
        self.plot_timer.setInterval(1000 / self.plot_update_freq)  # 10 Hz
        self.plot_timer.timeout.connect(self.update_plot)

        # Initialize data lists
        self.force_right_list = []
        self.force_left_list = []

    # Receive data from SerialReaderThread
    def receive_data(self, force_right, force_left):
        if force_right and force_left:            
            # Append new data with timestamps
            current_time = time.time()
            self.force_right_list.append((current_time, force_right))
            self.force_left_list.append((current_time, force_left))

            # Keep only the last 10 seconds of data
            self.force_right_list = [(t, val) for t, val in self.force_right_list if current_time - t <= self.display_window]
            self.force_left_list = [(t, val) for t, val in self.force_left_list if current_time - t <= self.display_window]

    def start_plotting(self):
        self.force_right_list = []
        self.force_left_list = []
        self.plot_timer.start()

    def update_plot(self):
        """Update the plot to show the last 10 seconds of data."""
        if len(self.force_left_list) > 0 and len(self.force_right_list) > 0:
            # Extract timestamps and values for plotting
            times_right = [t for t, _ in self.force_right_list]
            values_right = [val for _, val in self.force_right_list]
            times_left = [t for t, _ in self.force_left_list]
            values_left = [val for _, val in self.force_left_list]

            # Ensure times are sorted
            times_right, values_right = zip(*sorted(zip(times_right, values_right)))
            times_left, values_left = zip(*sorted(zip(times_left, values_left)))

            # Set x-axis range to the last 'display_window' seconds
            if times_right and times_left:
                start_time = max(times_right[0], times_left[0])
                end_time = times_right[-1] if times_right[-1] > times_left[-1] else times_left[-1]
                if start_time - end_time < 10:
                    self.plot_graph.setXRange(start_time, start_time+10)
                else:
                    self.plot_graph.setXRange(start_time, end_time)

            # Update the plots
            self.right_line.setData(times_right, values_right)
            self.left_line.setData(times_left, values_left)

            # Check for asymmetry and update background color
            raw_asym = abs(values_left[-1] - values_right[-1])
            # if max(values_left[-1],values_right[-1]) != 0:
            #     if (values_left[-1] > 30) & (values_right[-1] > 30):
            #         percent_asym = raw_asym / max(values_left[-1],values_right[-1])
            #         if percent_asym >= 0.10:
            #             self.plot_graph.setBackground("r")
            #         else:
            #             self.plot_graph.setBackground("w")
            #     else:
            #         self.plot_graph.setBackground("w")
            # else: 
            #     self.plot_graph.setBackground("w")


Main Window

In [ ]:
class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super().__init__()
        
        # Receive signal from SerialReaderThread
        self.serial_thread = SerialReaderThread()
        self.serial_thread.data_received.connect(self.update_function)
        self.serial_thread.error_message_signal.connect(self.show_error_message)

        # Initialize subwindows and receive signal from SerialReaderThread
        self.diffbar_window = diffbar_MainWindow()
        self.serial_thread.data_received.connect(self.diffbar_window.receive_data)

        self.diffline_window = diffline_MainWindow()
        self.serial_thread.data_received.connect(self.diffline_window.receive_data)

        self.absbar_window = absbar_MainWindow()
        self.serial_thread.data_received.connect(self.absbar_window.receive_data)

        self.absline_window = AbsLineMainWindow()
        self.serial_thread.data_received.connect(self.absline_window.receive_data)

        self.serial_thread.start()

        # Set up UI
        self.ui = Ui_MainWindow()
        self.ui.setupUi(self)
        
        self.ui.countdownLabel.hide()
        
        self.ui.absbar_button.clicked.connect(self.set_last_graph_clicked)
        self.ui.absline_button.clicked.connect(self.set_last_graph_clicked)
        self.ui.diffbar_button.clicked.connect(self.set_last_graph_clicked)
        self.ui.diffline_button.clicked.connect(self.set_last_graph_clicked)
        self.last_clicked_graph_button = None
        
        self.ui.tare_button.clicked.connect(self.tare)
        self.ui.start_button.clicked.connect(self.start)
        
        self.countdown_timer = None  # Initialize the countdown timer

    def update_function(self, right_list, left_list):
        pass

    def tare(self):
        self.serial_thread.tare()
        self.change_button_color()
        
    def set_last_graph_clicked(self):
        button = self.sender()
        if button:
            self.reset_graph_buttons()
            button.setStyleSheet("background-color: green; color: white;")
            self.last_clicked_graph_button = button
            print(self.last_clicked_graph_button.text())
    
    def start(self): 
        self.change_button_color()
        self.start_countdown(5)
        
    def start_countdown(self, seconds):
        self.countdown_time = seconds
        self.ui.countdownLabel.show()
        self.ui.countdownLabel.setText(f"Starting in: {self.countdown_time}")
        self.countdown_timer = QTimer(self)
        self.countdown_timer.timeout.connect(self.update_countdown)
        self.countdown_timer.start(1000)  # Update every second

    def update_countdown(self):
        self.countdown_time -= 1
        if self.countdown_time > 0:
            self.ui.countdownLabel.setText(f"Starting in: {self.countdown_time}")
        else:
            self.ui.countdownLabel.setText("Starting now!")
            self.countdown_timer.stop()  # Stop the countdown
            self.begin_graph()  # Start the graph after countdown ends
            

    def reset_graph_buttons(self):
        for button in [self.ui.absbar_button, self.ui.absline_button, self.ui.diffbar_button, self.ui.diffline_button]:
            button.setStyleSheet("background-color: white; color: black;")
            
    def set_unclicked(self):
        for button in [self.ui.absbar_button, self.ui.absline_button, self.ui.diffbar_button, 
                       self.ui.diffline_button, self.ui.tare_button, self.ui.start_button]:
            button.setStyleSheet("background-color: white; color: black;")  # Reset the style
            button.setChecked(False)   # This line is more relevant for toggle buttons

    def change_button_color(self):
        button = self.sender()
        if button: 
            button.setStyleSheet("background-color: green; color: white;")
    
    def begin_graph(self): 
        if self.last_clicked_graph_button.text() == 'Force Difference Bar Graph': 
            self.diffbar_window.start_plotting()
            self.diffbar_window.show()
        elif self.last_clicked_graph_button.text() == 'Force Difference Line Graph':
            self.diffline_window.start_plotting()
            self.diffline_window.show()
        elif self.last_clicked_graph_button.text() == 'Absolute Force Bar Graph':
            self.absbar_window.start_plotting()
            self.absbar_window.show()
        elif self.last_clicked_graph_button.text() == 'Absolute Force Line Graph':
            self.absline_window.start_plotting()
            self.absline_window.show()

        self.ui.countdownLabel.hide()
        self.set_unclicked()   

    def show_error_message(self, title, message):
        """
        Display an error message using QMessageBox.
        
        :param title: The title of the error message box.
        :param message: The message to be displayed.
        """
        msg_box = QMessageBox()
        msg_box.setIcon(QMessageBox.Critical)
        msg_box.setWindowTitle(title)
        msg_box.setText(message)
        msg_box.exec_()


In [ ]:
if __name__ == "__main__":
    app = QApplication(sys.argv)
    widget = MainWindow()
    widget.show()
    sys.exit(app.exec())